In [1]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

In [2]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [3]:
DATASET_SAVED = "../../data/raw_official"
URL_INDICATORS = "../../data/urls"

In [4]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [5]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [6]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'HCF_NO_ELECTRICITY', 'UHC_INDEX_REPORTED', 'UHC_AVAILABILITY_SCORE', 'HSS_COMPREHENSIVEHEALTHPOLICY', 'UHC_HRHSURGEONDENSITY', 'HRH_40', 'WHS6_15', 'HRH_33', 'HRH_26', 'WHS7_103', 'WHS6_102', 'HCF_UNREL_ELECTRICITY', 'HCF_REL_ELECTRICITY'}


In [7]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Dim1', 'Value', 'NumericValue', 'TimeDimensionBegin', 'TimeDimensionEnd', 'IndicatorCode']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)

    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [8]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [9]:
retrieve_all()

 54%|█████▍    | 7/13 [00:11<00:07,  1.28s/it]

Error: Expecting value: line 1 column 1 (char 0)


100%|██████████| 13/13 [00:22<00:00,  1.75s/it]


In [10]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "../../data/raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [11]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [12]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|██████████| 553/553 [00:00<00:00, 956.90it/s]


Tổng số điểm dữ liệu là: 453077
Hoàn thành xử lý: BMI


In [13]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|██████████| 553/553 [00:00<00:00, 1843.69it/s]


Tổng số điểm dữ liệu là: 211564
Hoàn thành xử lý: diabetes


In [14]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|██████████| 553/553 [00:01<00:00, 486.79it/s] 


Tổng số điểm dữ liệu là: 297817
Hoàn thành xử lý: alcohol_consumption


In [15]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

100%|██████████| 553/553 [00:00<00:00, 602.06it/s]


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [16]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|██████████| 553/553 [00:00<00:00, 3676.30it/s]


Tổng số điểm dữ liệu là: 141804
Hoàn thành xử lý: cholesterol


In [17]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████| 553/553 [00:00<00:00, 33467.28it/s]

Tổng số điểm dữ liệu là: 21630
Hoàn thành xử lý: glucose


In [18]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████| 553/553 [00:00<00:00, 18041.07it/s]

Tổng số điểm dữ liệu là: 35523


Hoàn thành xử lý: physical_activities


In [19]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|██████████| 553/553 [00:00<00:00, 10001.38it/s]

Tổng số điểm dữ liệu là: 8989
Hoàn thành xử lý: infrastructure


In [20]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|██████████| 553/553 [00:04<00:00, 135.41it/s] 


Tổng số điểm dữ liệu là: 707018
Hoàn thành xử lý: tobacco


In [21]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|██████████| 553/553 [00:00<00:00, 1729.79it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
